# Assignment — LangChain Fundamentals
## Landscape through Tools

**Domain for this assignment: GreenPlate — a restaurant reservation and food ordering
assistant.** This is a deliberately different scenario from CineBot, used throughout your
notebooks — the goal is to prove you understand the *concepts*, not that you can copy-paste
code you've already seen with new variable names.

**Structure:** Part A is conceptual (no coding), Part B is coding exercises, both arranged from
easier to harder. Part C is a single capstone challenge that combines everything. Attempt
sections in order — later questions build on ideas from earlier ones.

**Before you start:** make sure your environment is set up (API key loaded) and you can run a
basic `model.invoke()` successfully.

**A note on collaboration:** discussing concepts with classmates is encouraged. Copying code
without understanding it will be obvious the moment a follow-up question asks you to modify or
explain it.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file or Colab Secrets"

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5-mini")
print("Environment ready.")


---
# Part A — Conceptual Questions

## A1. Foundations (Easy)

1. In your own words, explain the sentence: *"An agent is a model calling tools in a loop until
   a task is complete. A harness is everything around that loop."* What specifically counts as
   part of the "harness"?

2. Name the four products in the Lang family (LangChain, LangGraph, LangSmith, Deep Agents) and
   state, in one sentence each, what job each one does. Which one is fundamentally different in
   *kind* from the other three, and why?

3. If you found a 2026-dated tutorial using `AgentExecutor` or `initialize_agent`, what would
   you conclude, and what should you use instead?

4. Why does a `.env` file exist? What specifically goes wrong if you skip it and hardcode an API
   key directly into a notebook cell?

5. What is the difference between a plain text prompt, a message-object list, and a
   dictionary-based message list? Give one situation where each is the natural choice.

## A2. Models, Messages, and Templates (Medium)

6. An `AIMessage` carries more than `.content`. Name at least four other fields or attributes it
   can carry, and what each one is actually useful for.

7. Explain why streaming produces `AIMessageChunk` objects instead of plain text fragments, and
   what property of these chunks makes them genuinely different from a string split into pieces.

8. What's the difference between `.batch()` and `.batch_as_completed()`? Describe a real
   situation where you would specifically want the second one over the first.

9. A `ToolMessage` has both a `.content` field and an `.artifact` field. What's the difference,
   and why would a RAG-style tool specifically want to use `.artifact`?

10. You're writing a `ChatPromptTemplate` whose system message needs to include a literal JSON
    example like `{"status": "ok"}`. What will go wrong if you paste that in directly, and how
    do you fix it?

11. What does `MessagesPlaceholder` do, and why can't you achieve the same result with a normal
    string-based template variable?

## A3. Structured Output (Medium-Hard)

12. Explain, in your own words, why asking a model to "please respond in JSON" via plain prompt
    instructions is fundamentally less reliable than using `with_structured_output()`.

13. What is `model.profile`, and how is it actually used internally when you call
    `with_structured_output()` without specifying a strategy explicitly?

14. A teammate writes:
    `model.with_structured_output(BookingRequest, strategy=ProviderStrategy(BookingRequest))`
    and asks you to code review it. What's wrong with this line, and what should it be instead?

15. What's the difference between structured output at the **model** level
    (`with_structured_output`) and at the **agent** level (`response_format` on `create_agent`)?
    Why does almost everything from Part 7 onward use the agent-level version?

16. You have a schema `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`.
    Explain what happens internally when the model receives a message that's genuinely
    ambiguous between the two schemas.

17. A schema field is defined as `party_size: int = Field(ge=1, le=20)`, and a customer message
    says "table for 50 please." Walk through, step by step, what happens inside the agent loop
    from the moment the model first proposes `party_size=50` to the moment a valid final answer
    is produced.

## A4. Tools (Hard)

18. A tool's docstring is described as "the tool's entire pitch to the model," not documentation
    for humans. Defend or challenge this claim — is there ever a case where the docstring
    genuinely doesn't matter much?

19. Explain what `ToolRuntime` actually hides from the model, and how LangChain knows to hide it
    (i.e., what specifically triggers this behavior)?

20. Compare `runtime.state`, `runtime.context`, and `runtime.store`. For each one, state: (a) how
    long the data persists, and (b) one concrete example of information that belongs there.

21. A tool accidentally declares a parameter named `config`. What actually happens when the
    agent tries to call it, and why is this a genuinely easy mistake to make by accident?

22. Explain the difference between a tool returning a plain string versus returning a `Command`.
    Give an original example (not from any notebook you've seen) of a situation that specifically
    requires `Command`, and explain why a plain string return wouldn't work for that case.

23. Describe, precisely, why `wrap_model_call`-based tool gating (making a tool invisible) is a
    stronger guarantee than instructing the model in the system prompt not to use a tool.

24. What is a **headless tool**, and how is its execution model fundamentally different from
    every other tool pattern covered in this course? Name one realistic capability that could
    only be implemented this way.


---
# Part B — Coding Exercises

All exercises use **GreenPlate**, a restaurant reservation and food ordering assistant. Each
question gives you a blank code cell to work in — write your solution there and run it to
confirm it works before moving on.

## B1. Easy

**B1.1 — A basic tool.** Write a `@tool`-decorated function called `check_table_availability`
that takes a `party_size: int` and a `time_slot: str`, and returns a string saying whether a
table is available (you can hardcode fake availability data, e.g. tables available for parties
of 2-6 at "7:00 PM" and "8:30 PM" only). Include a proper docstring. Print the tool's `.name`,
`.description`, and `.args` to confirm it's built correctly.


In [ ]:
# Your solution for B1.1


**B1.2 — A reusable prompt template.** Build a `ChatPromptTemplate` that generates a short,
enthusiastic description of a dish, given `{dish_name}` and `{cuisine_type}` as variables. Run
it with at least two different dish/cuisine combinations and print both results.


In [ ]:
# Your solution for B1.2


**B1.3 — A schema with a constrained field.** Define a Pydantic `BaseModel` called
`FoodOrder` with fields: `customer_name` (str), `dish_name` (str), `quantity` (int, must be
between 1 and 10), and `spice_level` (a `Literal` restricted to `"mild"`, `"medium"`, or
`"hot"`). Use `with_structured_output()` to extract a `FoodOrder` from this message: *"Hi, I'm
Karan, 2 butter chicken please, medium spice."* Print the result.


In [ ]:
# Your solution for B1.3


## B2. Medium

**B2.1 — Two schemas, one agent.** A restaurant assistant needs to handle both new reservations
and cancellations. Define `NewReservation` (customer_name, party_size, time_slot) and
`CancelReservation` (customer_name, time_slot) as separate Pydantic models. Build a
`create_agent` with `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`,
and test it with one clearly-a-reservation message and one clearly-a-cancellation message. Use
`isinstance()` to print which schema was chosen each time.


In [ ]:
# Your solution for B2.1


**B2.2 — A tool with `args_schema`.** Define a Pydantic input schema called `OrderInput` with
`dish_name` (str, with a description), `quantity` (int, `ge=1, le=10`, with a description), and
`delivery_or_pickup` (a `Literal["delivery", "pickup"]`, defaulting to `"pickup"`). Build a tool
called `place_order` using `args_schema=OrderInput`. Print `place_order.args` to confirm the
schema came through correctly, including the constraint and the default.


In [ ]:
# Your solution for B2.2


**B2.3 — Deliberately trigger a validation failure and watch self-correction.** Using the
`FoodOrder` schema from B1.3, build a `create_agent` with
`response_format=ToolStrategy(FoodOrder)`. Send a message asking for 15 units of a dish (this
should violate your `quantity` constraint). Print the full `result["messages"]` trace and point
out, in a comment, exactly where the self-correction happens.


In [ ]:
# Your solution for B2.3


## B3. Hard

**B3.1 — Long-term memory with `ToolRuntime`.** Build two tools: `save_dietary_preference`
(takes `customer_id`, `preference`, and `runtime: ToolRuntime`, saves to `runtime.store`) and
`recall_dietary_preference` (takes `customer_id` and `runtime: ToolRuntime`, reads it back).
Build an agent with these two tools and a real `store=` attached. Prove the memory survives
across two **separate** `.invoke()` calls — save a preference in the first call, and recall it
correctly in a second, independent call.


In [ ]:
# Your solution for B3.1


**B3.2 — Dynamic tool gating.** GreenPlate has a `book_private_dining_room` tool that should
only be available to customers with a "premium" membership tier. Using `wrap_model_call`, write
a middleware function that removes this tool from the model's visible toolset unless
`request.state.get("is_premium_member")` is `True`. Prove it works by running the SAME query
twice — once without the flag, once with it — and show the tool is genuinely unavailable in the
first case (not just "declined").


In [ ]:
# Your solution for B3.2


**B3.3 — Combine structured output and tools in one agent.** Build a `create_agent` that has
BOTH a `check_table_availability`-style tool AND a `response_format=ReservationConfirmation`
schema (define this schema yourself — it should capture at minimum: customer_name, time_slot,
confirmed: bool). Send a request that requires the agent to actually call the tool to check
availability *before* it can correctly fill in `confirmed`. Print both `result["messages"]` and
`result["structured_response"]`, and explain in a comment why this required the agent-level
`response_format`, not the raw model-level `with_structured_output()`.


In [ ]:
# Your solution for B3.3


---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for GreenPlate that combines **at least five** of the
following in one working system (your choice which five, but justify your choices in a markdown
cell before your code):

- A structured schema for orders or reservations (with at least one real constraint, like a
  `Literal` or a numeric range)
- At least two custom tools
- Long-term memory via `ToolRuntime.store` (e.g. remembering a customer's dietary preferences or
  favorite dish across sessions)
- Short-term memory via a checkpointer and `thread_id` (e.g. remembering the customer's name
  within one conversation)
- Dynamic tool gating based on some condition (membership tier, time of day, order size — your
  choice)
- A `context_schema` carrying some per-run data a tool reads (e.g. `restaurant_location`)

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls that demonstrate the system actually working — not just that
   it builds without error.
4. A short markdown reflection (3-5 sentences) on ONE trade-off or limitation of your design —
   what would break, or what would you need to add, if this went to real production use.

This is intentionally open-ended. There is no single correct architecture — the goal is
demonstrating you can combine these pieces into something coherent, not matching a hidden answer
key.


*Design explanation goes here (before your code):*


In [ ]:
# Your capstone solution


*Your reflection on trade-offs/limitations goes here:*
